# What still need to be done
- statistical analysis
- more optimization
- after tweaking hypothesis we need to run  on the posts fetched  from x
- charts
# issues 
- what type of metric we look for
- cma still not that good using text mining
- charts selection 
# feedback
put ur feedback here as bullets points 
- example : not all dataset is needed 

In [1]:
import pandas as pd
from transformers import pipeline , AutoTokenizer , AutoModelForSequenceClassification
import nltk , torch ,re
from nltk.corpus import wordnet, stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import pprint
#import os
import json

In [79]:
# this to dowload the dictionary needed by nltk runs ones 
#nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('maxent_ne_chunker_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [5]:
# vars only name  u can change 
name = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, dtype=torch.float32).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
# it  uses ur gpu or cpu  for torch 
model.to(device)
# We need this later  for calculating 
ent_idx = model.config.label2id["entailment"]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

In [64]:
#help lemitization  the text  
lem = WordNetLemmatizer()

In [65]:
# clensing text from number and pctuation 
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
stop_words = set(stopwords.words('english'))
 
def tokenize_and_filter(text):
    cleaned = clean_text(text)
    tokens = word_tokenize(cleaned.lower())
    return [w for w in tokens if w not in stop_words]
def expand(words):
    out = set(words)
    for w in words:
        for syn in wordnet.synsets(w, pos="n")[:2]:      
            out.update(l.name().lower() for l in syn.lemmas() if "_" not in l.name())
    return out


def to_lemma(tok):
    return lem.lemmatize(lem.lemmatize(tok, "v"), "n")

def frequences(tokens):
    lemmas = [to_lemma(t) for t in tokens]
    return {k: sum(l in words for l in lemmas) for k, words in lexicon.items()}

In [16]:
def score_text(text, hyps, batch_size=8):
    scores = []
    for i in range(0, len(hyps), batch_size):
        batch = hyps[i:i+batch_size]
        inputs = tok([text] * len(batch), batch,
                     return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits          # 3 numbers per pair
        probs = torch.softmax(logits.float(), dim=-1) 
        scores += probs[:, ent_idx].tolist()         # We keep only entailment
    return scores 
# agregation of frames value sum / numner  frame 
# example: 3 respensibility value -> we sum the three probability / 3  
#def agr_scores(df , columns):
#    norm_scores = pd.DataFrame(columns=columns)
#    for c in columns:
#        tmp = df[c]
#        if (len(tmp.shape) <= 1) :
#             norm_scores[c]= tmp    
#        else:
#            number_frames = tmp.shape[1] 
#            norm_scores[c]= tmp.sum(axis=1) / number_frames
#    return norm_scores

In [7]:
print(model.config.label2id) 
print(model.config.dtype) 

{'contradiction': 2, 'entailment': 0, 'neutral': 1}
torch.float32


In [31]:
df_cma_short = pd.DataFrame({
    "id":    ["CMA-WATER", "CMA-WAR", "CMA-PESTS", "CMA-JOURNEY", "CMA-BUILD"],
    "frame": ["WATER", "WAR", "PESTS/DISEASE", "JOURNEY", "BUILDINGS/STRUCTURE"],
    "hyp":   [
        "The text describes refugees as a flood or a wave.",
        "The text describes refugees as an invasion or an attack.",
        "The text describes refugees as vermin or a disease.",
        "The text describes migration as a journey or a path.",
        "The text talks about borders using words like wall, door or house.",
    ],
})

df_hyp_all = pd.concat([df_hyp_cleared, df_cma_short], ignore_index=True)

In [12]:
# chage name of file and sheet 's name  if they are diffrent in ur pc
#df_hypothesis = pd.read_excel("Human_pilot_llm_pipeline.xlsx",sheet_name="Hypothesis Bank")
df_tweets = pd.read_excel("refugee_tweets_deduplicated.xlsx")
with open("Human_pilot_llm_pipeline_FINAL_v3.json" , "r") as f:
    js = json.loads(f.read())

In [11]:
print(df_hyp_cleared["frame"].unique())

<StringArray>
[       'Responsibility',              'Conflict',        'Human Interest',
 'Economic Consequences',              'Morality']
Length: 5, dtype: str


In [10]:
hypothesis =js["hypothesis_bank"]
df_hyp= pd.DataFrame(hypothesis["sv_items"])
df_hyp_cleared = df_hyp[["id" , "frame" , "hyp"]]
df_hyp_cleared["frame"] = df_hyp_cleared["frame"].replace({"Economic": "Economic Consequences"})

In [43]:
hyp_ids    = df_hyp_all["id"].tolist()
hyp_frames = df_hyp_all["frame"].tolist()
hyp_texts  = df_hyp_all["hyp"].tolist()
#print(len(hyp_ids))
#print(df_hyp_all["frame"].value_counts())
#print(df_hyp_all.tail(6))
# removed repeated frames
# non_rep_frames = list(set(df_hyp_cleared['frame']))

In [33]:
# seperated per new york times article
ny_ranges=df_tweets[df_tweets["Handle"] == "@nytimes"].index
length_ny_ranges = len(ny_ranges)
tweets_separated_per_article = [
    df_tweets.iloc[ny_ranges[i]:ny_ranges[i + 1]]
    if i + 1 < len(ny_ranges)
    else df_tweets.iloc[ny_ranges[i]:]
    for i in range(0, len(ny_ranges), 1)
]

In [44]:
all_posts = pd.concat(tweets_separated_per_article, ignore_index=False)  # keeps original post ids
#print(all_posts["Set"].unique())  
syrian    = all_posts[all_posts["Set"] == "Syrian Refugee Crisis"]
ukrainian = all_posts[all_posts["Set"] == "Ukrainian Refugee Crisis"]
#print(len(syrian), len(ukrainian))   

In [45]:
syrian_articles    = [t for t in tweets_separated_per_article
                      if t["Set"].iloc[0] == "Syrian Refugee Crisis"]
ukrainian_articles = [t for t in tweets_separated_per_article
                      if t["Set"].iloc[0] == "Ukrainian Refugee Crisis"]
#print(len(syrian_articles), len(ukrainian_articles))   # number of articles in each

In [36]:
long_rows = []
for article_id, tweets in enumerate(tweets_separated_per_article):
    for post_id, row in tweets.iterrows():
        text = row["Text"]
        s = score_text(text, hyp_texts)
        for hid, fr, h, sc in zip(hyp_ids, hyp_frames, hyp_texts, s):
            long_rows.append({
                "set": row["Set"],
                "article_id": article_id,
                "post_id": post_id,
                "text": text,
                "hyp_id": hid,
                "frame": fr,
                "hypothesis": h,
                "score": round(sc, 3),
            })

long_df = pd.DataFrame(long_rows)

In [37]:
def make_final(d):
    frames_wide = (d.groupby(["post_id", "frame"])["score"].mean()
                    .unstack("frame").round(3))
    posts = (d[["set", "article_id", "post_id", "text"]]
             .drop_duplicates("post_id").set_index("post_id"))
    return posts.join(frames_wide)

final_syrian    = make_final(long_df[long_df["set"] == "Syrian Refugee Crisis"])
final_ukrainian = make_final(long_df[long_df["set"] == "Ukrainian Refugee Crisis"])

In [39]:
final_syrian.to_excel('syrian.xlsx', index=False)
final_ukrainian.to_csv('ukrainian.csv', index=False)

In [40]:
final_syrian.to_excel('syrian.xlsx', index=False)

In [41]:
#final.loc[81]["text"]

In [42]:
#long_df[long_df.post_id == 81][["hyp_id", "frame", "hypothesis", "score"]]

In [ ]:
# dictionary feel free to add/pop words from it
seeds = {
    "WATER":   ["flood", "wave", "tide", "flow", "stream", "trickle", "drown", "surge"],
    "WAR":     ["invasion", "attack", "battle", "fight", "defense", "siege", "assault"],
    "PESTS":   ["vermin", "infestation", "plague", "swarm", "disease", "contamination"],
    "JOURNEY": ["journey", "path", "route", "destination", "flight", "road"],
    "BUILD":   ["wall", "door", "house", "foundation", "gate", "fence"],
}
for  s in seeds.keys():
    res = expand(seeds[s])
    seeds[s] = res
#print("seed after expanding")
#pprint.pprint(seeds)

In [76]:
result_after_tok = []
for batch_tweets in tweets_separated_per_article:
    raw_texts = batch_tweets["Text"].tolist()
    result_after_tok.extend(tokenize_and_filter(t) for t in raw_texts)
print(len(result_after_tok)) 

86


In [77]:
# keep the originals, build the expanded lexicon separately
lexicon = {k: expand(v) for k, v in seeds.items()}
# always print and read the lists
p = [frequences(toks) for toks in result_after_tok]
cma_counts = pd.DataFrame(p)

In [ ]:
# total hits per domain
print(cma_counts.sum())

hits = cma_counts[cma_counts.sum(axis=1) > 0]
print(len(hits), "of", len(cma_counts), "posts have at least one hit")

# show the actual posts that matched
all_posts = pd.concat(tweets_separated_per_article, ignore_index=True)
for i in hits.index:
    print(i, hits.loc[i][hits.loc[i] > 0].to_dict(), "|", all_posts.loc[i, "Text"][:100])